# 02 — Civil war exposure
**Education Under Fire: Conflict Exposure and Educational Attainment in Nigeria**

Assigns each adult a war-exposure status from where they live and when they were born.

**Region.** The war region is the five Igbo-majority states that stayed under Biafran control for
most of the war: Abia, Anambra, Ebonyi, Enugu and Imo. The rest of the former Eastern Region and the
Mid-West were fought over for shorter periods, so they are *partly exposed*: excluded from the
comparison group in the main specification and added to the war region in a robustness check.

**Cohort.** A person is exposed if they were of school age (6–17) in at least one war year
(1967–69), i.e. born 1950–1963. `war_school_years` counts how many war years fell in their
school-age years (0–3). People born after 1963 were young children during the war and may have
been affected through health and nutrition, so the main comparison uses older cohorts only.

- **Input:** `data/processed/dhs_adults.csv`, state shapefile
- **Output:** `data/processed/analysis_war.csv`, `output/figures/fig_map_war_region.png`

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from config import *

adults = pd.read_csv(DHS_ADULTS)

## 1. Region groups

In [ ]:
adults["war_region"] = adults["state"].isin(WAR_STATES).astype(int)
adults["partly_exposed"] = adults["state"].isin(PARTLY_EXPOSED_STATES).astype(int)
adults["wider_war_region"] = (adults["war_region"] | adults["partly_exposed"]).astype(int)
adults["region_group"] = np.select(
    [adults["war_region"] == 1, adults["partly_exposed"] == 1],
    ["War region", "Partly exposed"], default="Rest of Nigeria",
)

assert set(WAR_STATES + PARTLY_EXPOSED_STATES) <= set(adults["state"]), "State name mismatch"
adults.groupby("region_group")["state"].unique()

## 2. Cohort exposure

In [ ]:
def school_age_years(birth_year, years):
    """Number of `years` in which someone born in `birth_year` was of school age."""
    ages = np.subtract.outer(np.asarray(years), birth_year.to_numpy())   # years x people
    return ((ages >= SCHOOL_AGE[0]) & (ages <= SCHOOL_AGE[1])).sum(axis=0)

adults["war_school_years"] = school_age_years(adults["birth_year"], WAR_YEARS)
adults["school_age_war"] = (adults["war_school_years"] > 0).astype(int)
adults["placebo_school_years"] = school_age_years(adults["birth_year"], PLACEBO_YEARS)
adults["school_age_placebo"] = (adults["placebo_school_years"] > 0).astype(int)

# Cohort groups relative to the war
adults["cohort_group"] = np.select(
    [adults["school_age_war"] == 1, adults["birth_year"] < 1950],
    ["School-age during war", "Finished school before war"],
    default="Young children during war or later",
)
adults["cohort5"] = adults["birth_year"] // 5 * 5

adults.groupby("cohort_group")["birth_year"].agg(["min", "max", "size"])

In [ ]:
# Main DiD sample: cohorts born before the war ended for school-age children (1935–63),
# war region vs rest of Nigeria (partly exposed states excluded)
adults["main_sample"] = (
    (adults["birth_year"] <= 1963) & (adults["partly_exposed"] == 0)
).astype(int)

adults["war_x_cohort"] = adults["war_region"] * adults["school_age_war"]
adults["war_x_years"] = adults["war_region"] * adults["war_school_years"]
adults["wider_x_cohort"] = adults["wider_war_region"] * adults["school_age_war"]

pd.crosstab([adults["cohort_group"]], adults["region_group"], margins=True)

## 3. Map of the war region

In [ ]:
states = gpd.read_file(STATE_SHAPEFILE)[["adm1_name", "geometry"]].rename(columns={"adm1_name": "state"})
states["group"] = np.select(
    [states["state"].isin(WAR_STATES), states["state"].isin(PARTLY_EXPOSED_STATES)],
    ["War region", "Partly exposed"], default="Rest of Nigeria",
)
colors = {"War region": "#B2182B", "Partly exposed": "#F4A582", "Rest of Nigeria": "#E0E0E0"}

fig, ax = plt.subplots(figsize=(8, 7))
for group, color in colors.items():
    states[states["group"] == group].plot(ax=ax, color=color, edgecolor="white", linewidth=0.6, label=group)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()]
ax.legend(handles, colors.keys(), frameon=False, loc="lower right")
ax.set_title("Nigerian Civil War (1967–70): exposure by present-day state")
ax.set_axis_off()
plt.tight_layout()
fig.savefig(FIGURES_DIR / "fig_map_war_region.png", dpi=300)

## 4. Save

In [ ]:
adults.to_csv(ANALYSIS_FILE, index=False)
print(f"Saved {len(adults):,} adults to {ANALYSIS_FILE.relative_to(PROJECT_ROOT)} "
      f"({adults['main_sample'].sum():,} in the main sample)")